# Cox Proportional Hazards

## Wstęp
&emsp;Model proporcjonalnego hazardu Cox'a jest modelem pozwalającym ocenić wpływ wielu czynników na funkcję hazardu (chwilowe natężenie ryzyka). W naszym przypadku oceniać będziemy wpływ na ryzyko zapłaty (zdarzenie).

&emsp; Przy budownie modelu uwzględnimy następujące cechy:
* `total_open_amount` *(kwota faktury)*
* `days_to_due` *(po ilu dniach płatność)*
* `invoice_age` *(wiek faktury)*
* `avg_delay_customer` *(średnie opóźnienie klienta)*
* `cust_payment_terms` *(warunki płatności)*
* `invoice_currency` *(waluta)*
* `segment` *(BE lub SME)*

### Biblioteki i przygotowanie danych
&emsp; Zacznijmy od zaimportowania niezbędnych narzędzi oraz przygotowania danych. Zostawymy jedynie kolumny odpowiadające interesującym nas cechom.

In [15]:
import pandas as pd
from lifelines import CoxPHFitter
import matplotlib.pyplot as plt

df = pd.read_csv('../data/dataset_survclean.csv')

features = [
    'total_open_amount',
    'days_to_due',
    'invoice_age',
    'avg_delay_customer',
    'cust_payment_terms',
    'invoice_currency',
    'segment',
    'time_days', 
    'event'
]

cox_df = df[features].copy()

cox_df = pd.get_dummies(cox_df, columns=['cust_payment_terms', 'invoice_currency', 'segment'], drop_first=True)
cox_df = cox_df.dropna()
cox_df = cox_df.astype(float)

display(cox_df.head())

,total_open_amount,days_to_due,invoice_age,avg_delay_customer,time_days,event,cust_payment_terms_B052,cust_payment_terms_BR12,cust_payment_terms_BR56,cust_payment_terms_C106,cust_payment_terms_C129,cust_payment_terms_CA10,cust_payment_terms_CA30,cust_payment_terms_CA60,cust_payment_terms_CAB1,cust_payment_terms_CAX2,cust_payment_terms_MC15,cust_payment_terms_NA10,cust_payment_terms_NA25,cust_payment_terms_NA31,cust_payment_terms_NA32,cust_payment_terms_NA38,cust_payment_terms_NA3B,cust_payment_terms_NA3F,cust_payment_terms_NA84,cust_payment_terms_NA8Q,cust_payment_terms_NA9X,cust_payment_terms_NAA8,cust_payment_terms_NAAW,cust_payment_terms_NAAX,cust_payment_terms_NAB1,cust_payment_terms_NABD,cust_payment_terms_NABG,cust_payment_terms_NAC6,cust_payment_terms_NACB,cust_payment_terms_NACE,cust_payment_terms_NACG,cust_payment_terms_NACH,cust_payment_terms_NAD1,cust_payment_terms_NAD4,...,cust_payment_terms_NAD8,cust_payment_terms_NAG2,cust_payment_terms_NAGD,cust_payment_terms_NAH4,cust_payment_terms_NAM1,cust_payment_terms_NAM2,cust_payment_terms_NAM3,cust_payment_terms_NAM4,cust_payment_terms_NATH,cust_payment_terms_NATJ,cust_payment_terms_NATK,cust_payment_terms_NATL,cust_payment_terms_NATM,cust_payment_terms_NATU,cust_payment_terms_NATV,cust_payment_terms_NATW,cust_payment_terms_NATX,cust_payment_terms_NATZ,cust_payment_terms_NAU5,cust_payment_terms_NAUP,cust_payment_terms_NAUW,cust_payment_terms_NAUY,cust_payment_terms_NAUZ,cust_payment_terms_NAV2,cust_payment_terms_NAV9,cust_payment_terms_NAVC,cust_payment_terms_NAVD,cust_payment_terms_NAVE,cust_payment_terms_NAVF,cust_payment_terms_NAVL,cust_payment_terms_NAVM,cust_payment_terms_NAVQ,cust_payment_terms_NAVR,cust_payment_terms_NAWM,cust_payment_terms_NAWN,cust_payment_terms_NAWP,cust_payment_terms_NAWU,cust_payment_terms_NAX2,invoice_currency_USD,segment_SME
0,54273.28,16.0,15.0,0.743688,17.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,79656.60,20.0,20.0,2.222222,17.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
2,2253.86,15.0,15.0,2.467532,107.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,3299.70,11.0,10.0,6.260714,53.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,33133.29,15.0,15.0,0.743688,12.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


### Model
&emsp; Pozostało dopasować model Coxa do badanego zbioru danych. Kolumną zawierającą informacje na temat upływu czasu jest `time_days`, a kolumną informującą o wystąpieniu zdarzenia jest kolumna `event`

In [16]:
cph = CoxPHFitter()

cph.fit(cox_df, duration_col='time_days', event_col='event')

/Users/bartekb/Bankrupt-Detector/venv/lib/python3.14/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['cust_payment_terms_B052', 'cust_payment_terms_BR12', 'cust_payment_terms_C129', 'cust_payment_terms_CA60', 'cust_payment_terms_MC15', 'cust_payment_terms_NA31', 'cust_payment_terms_NABD', 'cust_payment_terms_NACE', 'cust_payment_terms_NACG', 'cust_payment_terms_NAD8', 'cust_payment_terms_NATH', 'cust_payment_terms_NATK', 'cust_payment_terms_NATL', 'cust_payment_terms_NATU', 'cust_payment_terms_NATV', 'cust_payment_terms_NATW', 'cust_payment_terms_NATX', 'cust_payment_terms_NATZ', 'cust_payment_terms_NAUW', 'cust_payment_terms_NAUY', 'cust_payment_terms_NAV2', 'cust_payment_terms_NAV9', 'cust_payment_terms_NAWM'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarnin

<lifelines.CoxPHFitter: fitted with 48838 total observations, 9681 right-censored observations>

In [17]:
cph.print_summary()

<lifelines.CoxPHFitter: fitted with 48838 total observations, 9681 right-censored observations>
             duration col = 'time_days'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 48838
number of events observed = 39157
   partial log-likelihood = -393137.81
         time fit was run = 2026-07-28 17:08:02 UTC

---
                          coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                 
total_open_amount         0.00      1.00      0.00            0.00            0.00                1.00                1.00
days_to_due              -0.04      0.96      0.00           -0.04           -0.03                0.96                0.97
invoice_age              -0.10      0.90      0.03           -0.16           -0.05                0.85                0.96
avg_delay_customer       -0.02      0.98      0.00           -0.03           -0.02                0.97                0.98
cust_payment_terms_B052  -7.31      0.00      1.84          -10.93           -3.70                0.00                0.02
cust_payment_terms_BR12 -10.68      0.00      2.47          -15.53           -5.84                0.00                0.00
cust_payment_terms_BR56  -7.36      0.00      1.67          -10.63           -4.08                0.00                0.02
cust_payment_terms_C106 -11.83      0.00      2.78          -17.27           -6.38                0.00                0.00
cust_payment_terms_C129 -21.18      0.00    191.20         -395.93          353.57                0.00           3.59e+153
cust_payment_terms_CA10 -12.87      0.00      2.94          -18.63           -7.12                0.00                0.00
cust_payment_terms_CA30 -10.14      0.00      2.38          -14.80           -5.48                0.00                0.00
cust_payment_terms_CA60  -6.57      0.00      1.84          -10.17           -2.97                0.00                0.05
cust_payment_terms_CAB1 -13.54      0.00      3.24          -19.89           -7.19                0.00                0.00
cust_payment_terms_CAX2 -14.00      0.00      2.97          -19.82           -8.17                0.00                0.00
cust_payment_terms_MC15 -11.39      0.00      2.96          -17.20           -5.58                0.00                0.00
cust_payment_terms_NA10 -12.58      0.00      2.93          -18.33           -6.83                0.00                0.00
cust_payment_terms_NA25 -11.83      0.00      2.56          -16.84           -6.82                0.00                0.00
cust_payment_terms_NA31  -5.91      0.00      1.54           -8.93           -2.89                0.00                0.06
cust_payment_terms_NA32 -10.38      0.00      2.31          -14.91           -5.85                0.00                0.00
cust_payment_terms_NA38 -10.76      0.00      2.38          -15.42           -6.11                0.00                0.00
cust_payment_terms_NA3B -10.14      0.00      2.34          -14.73           -5.54                0.00                0.00
cust_payment_terms_NA3F -12.02      0.00      2.78          -17.47           -6.58                0.00                0.00
cust_payment_terms_NA84 -11.83      0.00      2.66          -17.06           -6.61                0.00                0.00
cust_payment_terms_NA8Q  -6.62      0.00      1.59           -9.75           -3.50                0.00                0.03
cust_payment_terms_NA9X -10.66      0.00      2.44          -15.45           -5.87                0.00                0.00
cust_payment_terms_NAA8 -12.40      0.00      2.79          -17.87           -6.93                0.00                0.00
cust_payment_terms_NAAW  -9.74      0.00      2.26          -14.16           -5.32                0.00                0.00
cust_payment_terms_NAAX -12.31      0.00      2.79          -